# ERC-20: data collection

Section 4.1 and Table 6. Four token lists define the labels:

| List | Class | Tokens in the paper |
|---|---|---|
| TokenScout, EOA-linked scam tokens (TO) | scam | 142,275 |
| ChainAbuse, tokens named in scam reports (CA) | scam | 11,636 |
| CoinMarketCap top-cap tokens (CMC) | legitimate | 2,003 |
| TokenScout academic benchmark, legitimate tokens (TS) | legitimate | 1,805 |

Each list is a CSV with `tokenname` and `contract_address` under `data/erc20/labels/`. For every contract the first 500 transfers are downloaded from Etherscan (ERC-20 transfers of the token, then native and internal ETH transfers of the contract, oldest first) and turned into graph descriptors (`src/bfx/erc20.py`).

Downloading needs an Etherscan API key in the environment variable `ETHERSCAN_API_KEY`; the cells are switched off by default.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from bfx import erc20

DATA = ROOT / "data" / "erc20"
DOWNLOAD = False
LISTS = {"tokenscout_scam_tokens": "tokenscout_scam.csv", "chainabuse_scam_tokens": "chainabuse_scam.csv",
         "cmc_licit_tokens": "cmc_licit.csv", "tokenscout_licit_tokens": "tokenscout_licit.csv"}

In [ ]:
if DOWNLOAD:
    for folder, file in LISTS.items():
        tokens = pd.read_csv(DATA / "labels" / file)
        for row in tokens.itertuples():
            target = DATA / "transfers" / folder / f"{row.tokenname}_{row.contract_address.lower()}.csv"
            if not target.exists():
                erc20.fetch_token_transfers(row.contract_address.lower(), target, max_transfers=500)

## Descriptors

The transfers of a token form a directed graph: nodes are addresses, an edge from $u$ to $v$ aggregates every transfer from $u$ to $v$ (total value, count, timestamps). Transfers before and after the median timestamp of the token are called long-term and short-term.

| Family | Descriptors |
|---|---|
| Structural | graph size (`num_nodes`, `num_edges`, `num_transactions`); degree, in-degree, out-degree, betweenness, closeness, eigenvector and Katz centrality; clustering coefficient; sender, receiver and address counts and ratios |
| Monetary | edge value (raw, normalised by token volume, log, harmonic); accumulated incoming and outgoing value and count per edge; long- and short-term average values per node; volume, value statistics and quantiles; Gini concentration of transfer values |
| Temporal | long- and short-term transfer counts per node and per edge; transfer frequency and recency per edge; time span; inter-arrival time statistics |

Node and edge descriptors are summarised by mean, standard deviation, minimum and maximum, which gives 146 numeric columns per token. Betweenness is estimated on 100 sampled pivots, so it varies slightly between runs.

When transfer files are present in `data/erc20/transfers/`, the cell below recomputes their descriptors and compares them with the stored feature table.

In [ ]:
stored = pd.read_csv(DATA / "features" / "cmc_licit_tokens.csv").set_index("token_file")
checks = []
for path in sorted((DATA / "transfers" / "cmc_licit_tokens").glob("*.csv"))[:3]:
    computed = pd.Series(erc20.token_features(erc20.read_transfers(path)))
    reference = stored.loc[path.name, computed.index].astype(float)
    exact = [c for c in computed.index if "betweenness" not in c]
    checks.append({"token": path.stem, "descriptors": len(computed),
                   "max_abs_difference": float(np.nanmax(np.abs(computed[exact].astype(float) - reference[exact])))})
pd.DataFrame(checks)

In [ ]:
if DOWNLOAD:
    for folder in LISTS:
        rows = []
        for path in sorted((DATA / "transfers" / folder).glob("*.csv")):
            transfers = erc20.read_transfers(path)
            if len(transfers):
                rows.append({"token_file": path.name, **erc20.token_features(transfers),
                             "label": "scam" if "scam" in folder else "licit"})
        pd.DataFrame(rows).to_csv(DATA / "features" / f"{folder}.csv", index=False)

Tokens with features per list:

In [ ]:
pd.Series({folder: sum(1 for _ in open(DATA / "features" / f"{folder}.csv")) - 1 for folder in LISTS},
          name="tokens")